In [3]:
"""
Y-Net style dual-decoder segmentation script (Kvasir-SEG).

CHANGES vs your original script (see chat discussion):
1. Decoder is now genuinely split into TWO independent branches
   (healthy_decoder, polyp_decoder), each with its own CBAM blocks.
   Only the encoder (with SE blocks) and skip connections are shared.
2. Model is now a proper Keras *multi-output* model
   (outputs: 'healthy_output', 'polyp_output'), so each branch gets
   its own loss, its own metrics, and can be monitored/checkpointed
   independently. This also removes the earlier ambiguity of whether
   both classes were "really" being learned separately.
3. Data generator now returns a dict {'healthy_output': ..., 'polyp_output': ...}
   instead of a single stacked (H, W, 2) array, to match the multi-output model.
4. Loss weighting: polyp branch is weighted higher (loss_weights) since it's
   the minority class and the metric you actually care about
   (val_polyp_output_dice_coef). Adjust PLOYP_LOSS_WEIGHT / HEALTHY_LOSS_WEIGHT
   below if you want to tune this.
5. Erosion behavior is UNCHANGED (kept intentionally, per your decision) --
   see the comment in `apply_erosion` / inside the generator for what it
   actually does to the boundary pixels.
"""

import os
import glob
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau


# ----------------------------------------------------------------------------
# Image / mask preprocessing utilities (unchanged from your script)
# ----------------------------------------------------------------------------
def resize_image(image, size=(256, 256)):
    """Resizes image or mask to target dimensions."""
    interpolation = cv2.INTER_AREA if len(image.shape) == 3 else cv2.INTER_NEAREST
    return cv2.resize(image, size, interpolation=interpolation)


def apply_clahe(image):
    """Applies Contrast Limited Adaptive Histogram Equalization (CLAHE)."""
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)


def apply_sharpening(image):
    """Applies a sharpening filter."""
    kernel = np.array([[0, -1, 0],
                        [-1, 5, -1],
                        [0, -1, 0]])
    return cv2.filter2D(image, -1, kernel)


def apply_gaussian_filter(image, kernel_size=(3, 3), sigma=0):
    """Applies Gaussian Blur for noise suppression."""
    return cv2.GaussianBlur(image, kernel_size, sigma)


def apply_erosion(mask, kernel_size=(3, 3)):
    """
    Applies morphological erosion to mask boundaries.

    NOTE (kept intentionally, per your decision): this is called
    independently on the healthy mask AND the polyp mask. Since the two
    masks are near-complementary, eroding both shrinks the boundary from
    *both* sides -> a ~2px wide band around every polyp boundary ends up
    labeled 0 in BOTH channels. This is NOT a true "ignore region"
    (it is not excluded from the loss) -- the model is still trained to
    predict 0 for both classes there. If you later want a real ignore
    region (excluded from the loss entirely), that needs an explicit
    sample-weight / mask passed into the loss, not just erosion.
    """
    kernel = np.ones(kernel_size, np.uint8)
    return cv2.erode(mask, kernel, iterations=1)


def apply_augmentations(image, mask_healthy, mask_polyp):
    """Applies synchronized spatial data augmentations."""
    if np.random.rand() > 0.5:
        image = cv2.flip(image, 1)
        mask_healthy = cv2.flip(mask_healthy, 1)
        mask_polyp = cv2.flip(mask_polyp, 1)

    if np.random.rand() > 0.5:
        image = cv2.flip(image, 0)
        mask_healthy = cv2.flip(mask_healthy, 0)
        mask_polyp = cv2.flip(mask_polyp, 0)

    return image, mask_healthy, mask_polyp


# ----------------------------------------------------------------------------
# Data generator -- now yields a DICT of masks to match the multi-output model
# ----------------------------------------------------------------------------
class KvasirDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, img_paths, img_dir, healthy_dir, polyp_dir,
                 batch_size=4, img_size=(256, 256), is_train=True):
        self.img_paths = img_paths
        self.img_dir = img_dir
        self.healthy_dir = healthy_dir
        self.polyp_dir = polyp_dir
        self.batch_size = batch_size
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self):
        return int(np.ceil(len(self.img_paths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_paths = self.img_paths[idx * self.batch_size: (idx + 1) * self.batch_size]

        batch_x = []
        batch_healthy = []
        batch_polyp = []

        for img_path in batch_paths:
            filename = os.path.basename(img_path)
            healthy_path = os.path.join(self.healthy_dir, filename)
            polyp_path = os.path.join(self.polyp_dir, filename)

            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            healthy_mask = cv2.imread(healthy_path, cv2.IMREAD_GRAYSCALE)
            polyp_mask = cv2.imread(polyp_path, cv2.IMREAD_GRAYSCALE)

            # 1. Resize
            image = resize_image(image, self.img_size)
            healthy_mask = resize_image(healthy_mask, self.img_size)
            polyp_mask = resize_image(polyp_mask, self.img_size)

            # 2. Image preprocessing
            image = apply_clahe(image)
            image = apply_sharpening(image)
            image = apply_gaussian_filter(image)

            # 3. Mask preprocessing (erosion kept as-is, see docstring above)
            healthy_mask = apply_erosion(healthy_mask)
            polyp_mask = apply_erosion(polyp_mask)

            # 4. Augmentations
            if self.is_train:
                image, healthy_mask, polyp_mask = apply_augmentations(
                    image, healthy_mask, polyp_mask
                )

            # Normalize image
            image = image.astype(np.float32) / 255.0

            # Binarize masks
            healthy_mask = (healthy_mask > 127).astype(np.float32)
            polyp_mask = (polyp_mask > 127).astype(np.float32)

            batch_x.append(image)
            batch_healthy.append(healthy_mask[..., np.newaxis])   # (H, W, 1)
            batch_polyp.append(polyp_mask[..., np.newaxis])       # (H, W, 1)

        return (
            np.array(batch_x),
            {
                "healthy_output": np.array(batch_healthy),
                "polyp_output": np.array(batch_polyp),
            },
        )


# ----------------------------------------------------------------------------
# Attention blocks
# ----------------------------------------------------------------------------
def squeeze_and_excitation(input_tensor, ratio=16):
    filters = input_tensor.shape[-1]
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(filters // ratio, activation="relu", use_bias=False)(se)
    se = layers.Dense(filters, activation="sigmoid", use_bias=False)(se)
    return layers.Multiply()([input_tensor, se])


def cbam_block(input_tensor, ratio=16):
    filters = input_tensor.shape[-1]

    # Channel attention
    shared_dense_1 = layers.Dense(filters // ratio, activation="relu", use_bias=False)
    shared_dense_2 = layers.Dense(filters, use_bias=False)

    avg_pool = layers.GlobalAveragePooling2D()(input_tensor)
    avg_pool = layers.Reshape((1, 1, filters))(avg_pool)
    avg_out = shared_dense_2(shared_dense_1(avg_pool))

    max_pool = layers.GlobalMaxPooling2D()(input_tensor)
    max_pool = layers.Reshape((1, 1, filters))(max_pool)
    max_out = shared_dense_2(shared_dense_1(max_pool))

    channel_attention = layers.Activation("sigmoid")(layers.Add()([avg_out, max_out]))
    x = layers.Multiply()([input_tensor, channel_attention])

    # Spatial attention
    avg_pool_sp = layers.Lambda(lambda z: tf.reduce_mean(z, axis=-1, keepdims=True))(x)
    max_pool_sp = layers.Lambda(lambda z: tf.reduce_max(z, axis=-1, keepdims=True))(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_sp, max_pool_sp])

    spatial_attention = layers.Conv2D(
        1, kernel_size=7, padding="same", activation="sigmoid", use_bias=False
    )(concat)
    return layers.Multiply()([x, spatial_attention])


# ----------------------------------------------------------------------------
# Encoder (shared) / Decoder (independent, called once per branch)
# ----------------------------------------------------------------------------
def se_encoder_block(inputs, filters):
    x = layers.Conv2D(filters, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = squeeze_and_excitation(x)
    p = layers.MaxPooling2D((2, 2))(x)
    return x, p


def cbam_decoder_block(inputs, skip_features, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=(2, 2), padding="same")(inputs)
    x = layers.Concatenate()([x, skip_features])

    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = cbam_block(x)
    return x


def build_dual_decoder_unet(input_shape=(256, 256, 3)):
    """
    Shared SE-encoder -> bottleneck -> TWO fully independent CBAM decoders
    (healthy_decoder, polyp_decoder). Each decoder has its own weights;
    only the skip connections (s1..s4) and bottleneck features are shared.
    """
    inputs = layers.Input(input_shape)

    # ---- Shared encoder (SE) ----
    s1, p1 = se_encoder_block(inputs, 64)
    s2, p2 = se_encoder_block(p1, 128)
    s3, p3 = se_encoder_block(p2, 256)
    s4, p4 = se_encoder_block(p3, 512)

    # ---- Shared bottleneck (SE) ----
    b1 = layers.Conv2D(1024, 3, padding="same")(p4)
    b1 = layers.BatchNormalization()(b1)
    b1 = layers.Activation("relu")(b1)
    b1 = layers.Conv2D(1024, 3, padding="same")(b1)
    b1 = layers.BatchNormalization()(b1)
    b1 = layers.Activation("relu")(b1)
    b1 = squeeze_and_excitation(b1)

    # ---- Healthy decoder branch (independent weights, CBAM) ----
    h1 = cbam_decoder_block(b1, s4, 512)
    h2 = cbam_decoder_block(h1, s3, 256)
    h3 = cbam_decoder_block(h2, s2, 128)
    h4 = cbam_decoder_block(h3, s1, 64)
    healthy_output = layers.Conv2D(
        1, 1, padding="same", activation="sigmoid", name="healthy_output"
    )(h4)

    # ---- Polyp decoder branch (independent weights, CBAM) ----
    d1 = cbam_decoder_block(b1, s4, 512)
    d2 = cbam_decoder_block(d1, s3, 256)
    d3 = cbam_decoder_block(d2, s2, 128)
    d4 = cbam_decoder_block(d3, s1, 64)
    polyp_output = layers.Conv2D(
        1, 1, padding="same", activation="sigmoid", name="polyp_output"
    )(d4)

    return models.Model(
        inputs, {"healthy_output": healthy_output, "polyp_output": polyp_output},
        name="Y_Net_Dual_Decoder_UNet",
    )



In [5]:

# ----------------------------------------------------------------------------
# Per-branch Dice metric (single-channel now, since branches are separate)
# ----------------------------------------------------------------------------
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def dice_bce_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice_loss = 1 - dice_coef(y_true, y_pred)
    return bce + dice_loss



In [12]:

model = build_dual_decoder_unet()
model.summary()
print(f"Total params: {model.count_params():,}")


# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
IMG_DIR = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG/images"
HEALTHY_DIR = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG/healthy-masks"
POLYP_DIR = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG/masks"
MODEL_SAVE_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/conference/claude/best_tf_dual_decoder_unet_v1.0.1.keras"

BATCH_SIZE = 4
EPOCHS = 100
LR = 1e-4

# Polyp is the minority / harder class and is the metric you actually care
# about (classification decision is derived from it) -> weight it higher.
HEALTHY_LOSS_WEIGHT = 1.0
POLYP_LOSS_WEIGHT = 1.5

# ----------------------------------------------------------------------------
# Data splitting (80% Train, 10% Val, 10% Test)
# ----------------------------------------------------------------------------
all_images = sorted(glob.glob(os.path.join(IMG_DIR, "*.[pj][pn][g]")))
np.random.shuffle(all_images)

total_len = len(all_images)
train_idx = int(0.8 * total_len)
val_idx = int(0.9 * total_len)

train_imgs = all_images[:train_idx]
val_imgs = all_images[train_idx:val_idx]
test_imgs = all_images[val_idx:]

print(f"Total Images: {total_len} | Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

train_gen = KvasirDataGenerator(train_imgs, IMG_DIR, HEALTHY_DIR, POLYP_DIR, batch_size=BATCH_SIZE, is_train=True)
val_gen = KvasirDataGenerator(val_imgs, IMG_DIR, HEALTHY_DIR, POLYP_DIR, batch_size=BATCH_SIZE, is_train=False)
test_gen = KvasirDataGenerator(test_imgs, IMG_DIR, HEALTHY_DIR, POLYP_DIR, batch_size=BATCH_SIZE, is_train=False)

# ----------------------------------------------------------------------------
# Compile (per-output losses, loss weights, metrics)
# ----------------------------------------------------------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss={
        "healthy_output": dice_bce_loss,
        "polyp_output": dice_bce_loss,
    },
    loss_weights={
        "healthy_output": HEALTHY_LOSS_WEIGHT,
        "polyp_output": POLYP_LOSS_WEIGHT,
    },
    metrics={
        "healthy_output": [dice_coef, tf.keras.metrics.Precision(name="precision"),
                            tf.keras.metrics.Recall(name="recall")],
        "polyp_output": [dice_coef, tf.keras.metrics.Precision(name="precision"),
                          tf.keras.metrics.Recall(name="recall")],
    },
)

# NOTE: with dict-style metrics, Keras names the logged metric
# "val_polyp_output_dice_coef" / "val_healthy_output_dice_coef".
callbacks = [
    ModelCheckpoint(MODEL_SAVE_PATH, monitor="val_polyp_output_dice_coef",
                     mode="max", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_polyp_output_dice_coef", mode="max",
                       factor=0.5, patience=5, min_lr=1e-6, verbose=1),
]

# ----------------------------------------------------------------------------
# Train
# ----------------------------------------------------------------------------
print(f"Targeting >85% Polyp DSC. Starting training for {EPOCHS} epochs...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
)

# ----------------------------------------------------------------------------
# Evaluate
# ----------------------------------------------------------------------------
print("\n--- Training Complete. Evaluating on Test Set ---")
model.load_weights(MODEL_SAVE_PATH)
test_metrics = model.evaluate(test_gen, verbose=1)
print(dict(zip(model.metrics_names, test_metrics)))

Model: "Y_Net_Dual_Decoder_UNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_91 (Conv2D)  │ (None, 256, 256,  │      1,792 │ input_layer_3[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_91[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_90       │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_92 (Conv2D)  │ (None, 256, 256,  │     36,928 │ activation_90[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_92[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_91       │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ activation_91[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_55          │ (None, 1, 1, 64)  │          0 │ global_average_p… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_70 (Dense)    │ (None, 1, 1, 4)   │        256 │ reshape_55[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_71 (Dense)    │ (None, 1, 1, 64)  │        256 │ dense_70[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_55         │ (None, 256, 256,  │          0 │ activation_91[0]… │
│ (Multiply)          │ 64)               │            │ dense_71[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_12    │ (None, 128, 128,  │          0 │ multiply_55[0][0] │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_93 (Conv2D)  │ (None, 128, 128,  │     73,856 │ max_pooling2d_12… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        512 │ conv2d_93[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_92       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_94 (Conv2D)  │ (None, 128, 128,  │    147,584 │ activation_92[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 43,513,938 (165.99 MB)

 Trainable params: 43,498,322 (165.93 MB)

 Non-trainable params: 15,616 (61.00 KB)

Total params: 43,513,938
Total Images: 1000 | Train: 800 | Val: 100 | Test: 100
Targeting >85% Polyp DSC. Starting training for 100 epochs...
Epoch 1/100


I0000 00:00:1787197681.997923   10376 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_90485__.418
I0000 00:00:1787197710.463979   10376 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion', 20 bytes spill stores, 20 bytes spill loads



200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - healthy_output_dice_coef: 0.7595 - healthy_output_loss: 0.7256 - healthy_output_precision: 0.8793 - healthy_output_recall: 0.8379 - loss: 2.5417 - polyp_output_dice_coef: 0.3079 - polyp_output_loss: 1.2107 - polyp_output_precision: 0.3291 - polyp_output_recall: 0.5896

I0000 00:00:1787197768.730635   10376 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_95734__.70



Epoch 1: val_polyp_output_dice_coef improved from None to 0.20068, saving model to /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/conference/claude/best_tf_dual_decoder_unet_v1.0.1.keras

Epoch 1: finished saving model to /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/conference/claude/best_tf_dual_decoder_unet_v1.0.1.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 134s 367ms/step - healthy_output_dice_coef: 0.8058 - healthy_output_loss: 0.6057 - healthy_output_precision: 0.8912 - healthy_output_recall: 0.8945 - loss: 2.2531 - polyp_output_dice_coef: 0.3487 - polyp_output_loss: 1.0983 - polyp_output_precision: 0.3903 - polyp_output_recall: 0.6045 - val_healthy_output_dice_coef: 0.6625 - val_healthy_output_loss: 0.9659 - val_healthy_output_precision: 0.8341 - val_healthy_output_recall: 0.9935 - val_loss: 2.9703 - val_polyp_output_dice_coef: 0.2007 - val_polyp_output_loss: 1.3363 - val_polyp_output_precision: 0.0000e+00 - val_polyp_output_recall: 0.0000e

In [2]:

def plot_training_curves(hist, save_path=None):
    """hist: a dict like history.history (or one loaded back from pickle)."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    pairs = [
        ("loss", "val_loss", "Total loss (weighted sum)"),
        ("healthy_output_loss", "val_healthy_output_loss", "Healthy branch loss"),
        ("polyp_output_loss", "val_polyp_output_loss", "Polyp branch loss"),
        ("healthy_output_dice_coef", "val_healthy_output_dice_coef", "Healthy Dice"),
        ("polyp_output_dice_coef", "val_polyp_output_dice_coef", "Polyp Dice"),
    ]

    for ax, (train_key, val_key, title) in zip(axes.flat, pairs):
        if train_key not in hist:
            ax.axis("off")
            continue
        epochs_range = range(1, len(hist[train_key]) + 1)
        ax.plot(epochs_range, hist[train_key], label="train")
        if val_key in hist:
            ax.plot(epochs_range, hist[val_key], label="val")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)

    # last subplot: learning rate, if tracked
    ax = axes.flat[5]
    if "learning_rate" in hist:
        ax.plot(range(1, len(hist["learning_rate"]) + 1), hist["learning_rate"])
        ax.set_title("Learning rate")
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.3)
    else:
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved training curves to: {save_path}")
    plt.show()
MODEL_SAVE_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/conference/claude/best_tf_dual_decoder_unet_v1.0.1.keras"
CURVES_SAVE_PATH = MODEL_SAVE_PATH.replace(".keras", "_curves.png")
plot_training_curves(history.history, save_path=CURVES_SAVE_PATH)

# ----------------------------------------------------------------------------
# Evaluate
# ----------------------------------------------------------------------------
print("\n--- Training Complete. Evaluating on Test Set ---")
model.load_weights(MODEL_SAVE_PATH)
# return_dict=True gives correctly-labeled results directly -- avoids the
# earlier zip(model.metrics_names, test_metrics) mismatch bug in Keras 3
# multi-output models (metrics_names doesn't always expand cleanly).
test_results = model.evaluate(test_gen, verbose=1, return_dict=True)
print(test_results)


NameError: name 'history' is not defined